In [0]:
from pyspark.sql import functions as F
RAW_TABLE = "echochain.bronze.raw_listings_2"
OUT_TABLE = "echochain.bronze.cleaned_listings_2"
raw = spark.table(RAW_TABLE)

In [0]:
print(raw.columns)

In [0]:
# 1. Map + rename columns per the mapping screenshot.
# Columns with NO equivalent in this dataset (listing_id, listing_url, scraped_at, bids, watchers, shipping_price, title) are handled separately in step 5 below — they don't exist on the raw table so there's nothing to select here.

mapped = raw.select(
    F.col("Brand").alias("brand"),
    F.col("Condition").alias("condition"),
    F.col("Country_Region_Of_Manufacture").alias("location"),
    F.col("Model").alias("model"),
    F.col("Price").alias("price_raw"),
    F.col("Processor").alias("processor"),
    F.col("Ram_Size").alias("ram_gb_raw"),
    F.col("Screen_Size").alias("screen_size_raw"),
    F.col("Seller_Note").alias("seller_notes"),
    F.col("SSD_Capacity").alias("ssd_gb_raw"),
)

In [0]:
# 2. Parse messy numeric fields.
#ram_gb/ssd_gb/screen_size_in -- pull the number out of the fields for a value

# price: sometimes a range ("$399.99 to $634.99") -> average of the range
mapped = mapped.withColumn(
    "price_low_extracted",
    F.regexp_extract(F.regexp_replace(F.col("price_raw"), ",", ""), r"\$?([\d.]+)", 1)
).withColumn(
    "price_high_extracted",
    F.regexp_extract(F.regexp_replace(F.col("price_raw"), ",", ""), r"to\s*\$?([\d.]+)", 1)
).withColumn(
    "price_low", F.expr("try_cast(price_low_extracted as double)")
).withColumn(
    "price_high", F.expr("try_cast(price_high_extracted as double)")
).withColumn(
    "price",
    F.when(F.col("price_high").isNotNull(), (F.col("price_low") + F.col("price_high")) / 2)
     .otherwise(F.col("price_low"))
)

#matching only GB suffixes for ram
# try_cast (not cast): regexp_extract returns "" on no match, not null, and a plain .cast("int") on "" errors instead of returning null.
mapped = mapped.withColumn(
    "ram_gb_extracted", F.regexp_extract(F.col("ram_gb_raw"), r"(\d+)\s*GB", 1)
).withColumn(
    "ram_gb", F.expr("try_cast(ram_gb_extracted as int)")
)

# ssd_gb: "512 GB" -> 512, "1 TB" / embedded "...Max 1TB..." -> 1000, "NO HDD" -> null
mapped = mapped.withColumn(
    "ssd_tb_extracted", F.regexp_extract(F.col("ssd_gb_raw"), r"(\d+(?:\.\d+)?)\s*TB", 1)
).withColumn(
    "ssd_gb_extracted", F.regexp_extract(F.col("ssd_gb_raw"), r"(\d+)\s*GB", 1)
).withColumn(
    "ssd_gb",
    F.when(F.upper(F.col("ssd_gb_raw")).contains("NO HDD"), F.lit(None).cast("int"))
     .when(F.upper(F.col("ssd_gb_raw")).rlike(r"\d+(\.\d+)?\s*TB"),
           (F.expr("try_cast(ssd_tb_extracted as double)") * 1000).cast("int"))
     .otherwise(F.expr("try_cast(ssd_gb_extracted as int)"))
)

# screen_size_in: "11.6 in", "11.6 inin" (typo), 'Minimum 12.5"' -> leading float, ignoring text around it
mapped = mapped.withColumn(
    "screen_size_extracted", F.regexp_extract(F.col("screen_size_raw"), r"(\d+(?:\.\d+)?)", 1)
).withColumn(
    "screen_size_in", F.expr("try_cast(screen_size_extracted as double)")
)

In [0]:
# 3. Text normalization — same pattern as the Week 2 job : lower + trim for brand and model
mapped = (
    mapped.withColumn("brand_clean", F.trim(F.lower(F.col("brand"))))
          .withColumn("model_clean", F.trim(F.lower(F.col("model"))))
)

In [0]:
# 4. damage_flag — same regex vocab as Week 2. That job ran it against title + seller_notes; this dataset has no raw title, so it runs against condition + seller_notes instead (condition here often carries long free-text descriptions, which is where this signal mostly comes from).
damage_regex = r"(?i)(crack|broken|hinge|dead pixel|for parts|spares|as is|repair|flicker|no boot|not work)"
 
mapped = (
    mapped.withColumn(
        "combined_text",
        F.concat_ws(" ", F.coalesce(F.col("condition"), F.lit("")), F.coalesce(F.col("seller_notes"), F.lit("")))
    )
    .withColumn(
        "damage_flag",
        F.when(F.col("combined_text").rlike(damage_regex), F.lit("likely_damaged")).otherwise(F.lit("unknown"))
    )
)

In [0]:
# 5. Columns with NO source in this dataset at all.
mapped = (
    # listing_id: synthetic — no id field exists in the Kaggle export.
    # Prefixed KAGGLE- so ids can never collide with the original dataset's.
    # (monotonically_increasing_id() was silently returning null on serverless compute — same class of issue as sparkContext.broadcast() earlier. uuid() is a plain SQL builtin with no such dependency.)
    mapped.withColumn("listing_id", F.concat(F.lit("KAGGLE-"), F.expr("uuid()")))
    # listing_url: no real URL exists for a static export — left null rather than fabricating a fake, clickable-looking eBay link.
    .withColumn("listing_url", F.lit(None).cast("string"))
    # scraped_at: this is a bulk load, not a live scrape — set to today's load date for every row. It's a load-date placeholder, not a real per-listing scrape timestamp.
    .withColumn("scraped_at", F.current_date())
    # bids / watchers / shipping_price: no equivalent field in this dataset.
    .withColumn("bids", F.lit(None).cast("int"))
    .withColumn("watchers", F.lit(None).cast("int"))
    .withColumn("shipping_price", F.lit(None).cast("double"))
    # title_clean: no raw title text exists to clean — left null rather than fabricating listing-title-sounding text. Not used by the Week 3 fuzzy matcher (that reads product_signature), so this is safe to leave null; it just won't be usable for anything that specifically needs listing title text.
    .withColumn("title_clean", F.lit(None).cast("string"))
)

In [0]:
# 6. product_signature — identical formula to Week 2, so the Week 3 fuzzy-matching script still works against this table unchanged.
mapped = (
    mapped.withColumn(
        "ram_token", F.when(F.col("ram_gb").isNotNull(), F.concat(F.col("ram_gb").cast("string"), F.lit("gb")))
    )
    .withColumn(
        "ssd_token", F.when(F.col("ssd_gb").isNotNull(), F.concat(F.col("ssd_gb").cast("string"), F.lit("gb")))
    )
    .withColumn(
        "product_signature",
        F.trim(F.concat_ws(" ", F.col("brand_clean"), F.col("model_clean"), F.col("ram_token"), F.col("ssd_token")))
    )
)

In [0]:
# 7. Final column set — EXACT same names/order as echochain.bronze.cleaned_listings
cleaned_listings_2 = mapped.select(
    "listing_id",
    "listing_url",
    "title_clean",
    "seller_notes",
    "condition",
    "damage_flag",
    "price",
    "shipping_price",
    "bids",
    "watchers",
    "location",
    "brand_clean",
    "model_clean",
    F.col("processor").alias("processor_final"),
    "ram_gb",
    "ssd_gb",
    "screen_size_in",
    "product_signature",
    "scraped_at",
)

In [0]:
# 8. Null check + duplicate check before writing
 
print("Null counts per column:")
null_counts = cleaned_listings_2.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cleaned_listings_2.columns]
)
display(null_counts)


dedup_cols = [c for c in cleaned_listings_2.columns if c != "listing_id"]
 
total_rows = cleaned_listings_2.count()
cleaned_listings_2 = cleaned_listings_2.dropDuplicates(dedup_cols)
distinct_rows = cleaned_listings_2.count()
print(f"Total rows: {total_rows}")
print(f"Distinct rows (all columns except listing_id): {distinct_rows}")
print(f"Duplicates dropped: {total_rows - distinct_rows}")

In [0]:
# 9. Write to Bronze
cleaned_listings_2.write.format("delta").mode("overwrite").saveAsTable(OUT_TABLE)
display(cleaned_listings_2)